# Exploring the WellConverge synthetic health data

Scratch space for adhoc analysis. Everything here is **synthetic** — generated by
`wellconverge_tools.datagen`, containing no real patient data.

Prerequisite: `wc-tools generate` (local), or `wc-tools deploy` if you want to read from S3.

Start the kernel with:

```bash
cd tools && uv run --extra notebook jupyter lab
```

In [ ]:
import pandas as pd

from wellconverge_tools.analysis import build_feature_table, load_all, top_coefficients, train_baseline

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

# source="local" reads tools/out; source="s3" reads the data lake directly.
frames = load_all(source="local")
patients, encounters, observations = frames["patients"], frames["encounters"], frames["observations"]

{name: frame.shape for name, frame in frames.items()}

## Cohort shape

In [ ]:
patients[["age_years", "bmi", "chronic_conditions"]].describe().round(2)

In [ ]:
pd.crosstab(patients["insurance_type"], patients["age_years"] >= 65, normalize="columns").round(3)

## Where readmission risk concentrates

In [ ]:
(
    encounters.groupby("primary_diagnosis_desc")
    .agg(encounters=("encounter_id", "count"),
         readmit_rate=("readmitted_30d", "mean"),
         median_los=("length_of_stay_days", "median"),
         median_charge=("total_charge_usd", "median"))
    .sort_values("readmit_rate", ascending=False)
    .round(3)
)

In [ ]:
# Abnormal labs vs. readmission — the strongest single clinical signal in the data.
abnormal = (
    observations.assign(is_abnormal=observations["abnormal_flag"].ne("N"))
    .groupby("encounter_id")["is_abnormal"].sum()
    .rename("abnormal_observations")
)
encounters.join(abnormal, on="encounter_id").groupby("abnormal_observations")["readmitted_30d"].agg(["mean", "count"]).round(3)

## Baseline model

In [ ]:
features = build_feature_table(patients, encounters, observations)
result = train_baseline(features)
print(result)
top_coefficients(result, n=15)

## Querying the catalog instead

Once `wc-tools catalog` has run, the same data is queryable through Glue/Athena —
which is what SageMaker Studio, Data Wrangler and Feature Store read from.

In [ ]:
from wellconverge_tools.awsio import athena, guarded_session
from wellconverge_tools.config import load_settings

settings = load_settings()
session = guarded_session(settings)

pd.DataFrame(athena.run_query(session, settings, """
    SELECT year, encounter_class, count(*) AS encounters,
           round(avg(CASE WHEN readmitted_30d THEN 1.0 ELSE 0.0 END), 3) AS readmit_rate
    FROM encounters
    GROUP BY year, encounter_class
    ORDER BY year, encounter_class
"""))